# E-Commerce EDA - Full Demo Project
This notebook covers loading, cleaning, univariate, bivariate, multivariate analysis.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10,5)

# Load
orders = pd.read_csv('../data/raw/demo_orders.csv')
customers = pd.read_csv('../data/raw/demo_customers.csv')
products = pd.read_csv('../data/raw/demo_products.csv')

print(f"Orders: {orders.shape}, Customers: {customers.shape}, Products: {products.shape}")
orders.head()


In [ ]:

# Data Understanding
print("--- Orders Info ---")
print(orders.info())
print("\n--- Missing Values ---")
print(orders.isnull().sum())
print("\n--- Duplicates ---", orders.duplicated().sum())


In [ ]:

# Cleaning
# Convert dates
orders['order_date'] = pd.to_datetime(orders['order_date'])
customers['signup_date'] = pd.to_datetime(customers['signup_date'])

# Fill missing delivery_days with median, rating with mode
orders['delivery_days'].fillna(orders['delivery_days'].median(), inplace=True)
orders['rating'].fillna(orders['rating'].mode()[0], inplace=True)

# Merge to create enriched dataset
df = orders.merge(customers, on='customer_id', how='left').merge(products, on='product_id', how='left')
df.head()


In [ ]:

# Univariate Analysis
fig, axes = plt.subplots(1,3, figsize=(15,4))
sns.histplot(df['total_amount'], bins=30, ax=axes[0], kde=True); axes[0].set_title('Total Amount Distribution')
sns.countplot(x='payment_method', data=df, ax=axes[1]); axes[1].set_title('Payment Method')
sns.countplot(x='category', data=df, ax=axes[2]); axes[2].tick_params(axis='x', rotation=45); axes[2].set_title('Category')
plt.tight_layout(); plt.show()

df['rating'].value_counts().sort_index().plot(kind='bar', title='Rating Distribution')
plt.show()


In [ ]:

# Bivariate Analysis
# 1. Delivery vs Rating
plt.figure()
sns.boxplot(x='rating', y='delivery_days', data=df)
plt.title('Does faster delivery = better rating?'); plt.show()

# 2. City vs Revenue
city_rev = df.groupby('city')['total_amount'].sum().sort_values(ascending=False)
city_rev.plot(kind='bar', title='Revenue by City'); plt.show()

# 3. Segment Analysis
sns.barplot(x='segment', y='total_amount', data=df, estimator=sum, ci=None)
plt.title('Revenue by Customer Segment'); plt.show()


In [ ]:

# Multivariate / Time Series
df['month'] = df['order_date'].dt.to_period('M').astype(str)
monthly = df.groupby('month')['total_amount'].sum()
monthly.plot(marker='o', figsize=(12,4), title='Monthly Revenue Trend')
plt.xticks(rotation=45); plt.show()

# Correlation
numeric_cols = ['quantity','delivery_days','rating','total_amount','price','age']
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap'); plt.show()


In [ ]:

# 5 Business Questions Answered
print("Q1: Premium vs Regular AOV")
print(df.groupby('segment')['total_amount'].agg(['mean','count']))

print("\nQ2: Category Profitability")
df['profit'] = (df['price_x'] - df['cost']) * df['quantity'] if 'cost' in df.columns else df['total_amount']*0.4
# Recalculate properly
df_merged_profit = df.copy()
df_merged_profit['profit'] = (df_merged_profit['price_y'] - df_merged_profit['price_y']*0.6) * df_merged_profit['quantity']
print(df_merged_profit.groupby('category')['profit'].sum().sort_values(ascending=False))

print("\nQ3: Payment method AOV")
print(df.groupby('payment_method')['total_amount'].mean())

# Save cleaned data
df.to_csv('../data/processed/cleaned_enriched.csv', index=False)
print("\nSaved cleaned_enriched.csv to data/processed/")


## Key Insights & Next Steps
1. Premium customers drive disproportionate revenue - focus retention
2. Delivery >4 days kills rating - optimize logistics
3. Card users have highest AOV - push Card offers
4. Build RFM model next
5. Build churn prediction as next project